# Ch05 练习参考答案：Double Q-learning

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils import set_seed
from rlenvs import CliffWalk

set_seed(0)


def epsilon_greedy_action(Q, s, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(Q.shape[1])
    return int(np.argmax(Q[s]))


def double_q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    nS, nA = env.nS, env.nA
    Q1 = np.zeros((nS, nA))
    Q2 = np.zeros((nS, nA))
    rewards = []
    for ep in range(n_episodes):
        s = env.reset()
        done = False
        ep_r = 0.0
        while not done:
            a = epsilon_greedy_action(Q1 + Q2, s, epsilon)
            s_next, r, done, _ = env.step(a)
            ep_r += r
            if np.random.random() < 0.5:
                a_star = int(np.argmax(Q1[s_next])) if not done else 0
                td_target = r + (0 if done else gamma * Q2[s_next, a_star])
                Q1[s, a] += alpha * (td_target - Q1[s, a])
            else:
                a_star = int(np.argmax(Q2[s_next])) if not done else 0
                td_target = r + (0 if done else gamma * Q1[s_next, a_star])
                Q2[s, a] += alpha * (td_target - Q2[s, a])
            s = s_next
        rewards.append(ep_r)
    return Q1 + Q2, np.array(rewards)


def q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    nS, nA = env.nS, env.nA
    Q = np.zeros((nS, nA))
    rewards = []
    for ep in range(n_episodes):
        s = env.reset()
        done = False
        ep_r = 0.0
        while not done:
            a = epsilon_greedy_action(Q, s, epsilon)
            s_next, r, done, _ = env.step(a)
            ep_r += r
            td_target = r + (0 if done else gamma * np.max(Q[s_next]))
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s_next
        rewards.append(ep_r)
    return Q, np.array(rewards)


# 1) 在 OneStateTrap 上验证消除 maximization bias
class OneStateTrap:
    def __init__(self):
        self.nS = 1
        self.nA = 2
        self._rng = np.random.default_rng()
    def reset(self):
        return 0
    def step(self, a):
        r = self._rng.normal(0, 1) if a == 0 else self._rng.normal(-0.1, 1)
        return 0, r, True, {}


def run_trap_double(n_episodes=300, alpha=0.1, epsilon=0.1, n_seeds=200):
    freqs = np.zeros((n_seeds, n_episodes))
    for seed in range(n_seeds):
        np.random.seed(seed)
        env = OneStateTrap()
        Q1 = np.zeros((1, 2))
        Q2 = np.zeros((1, 2))
        for ep in range(n_episodes):
            s = env.reset()
            done = False
            actions = []
            while not done:
                a = epsilon_greedy_action(Q1 + Q2, s, epsilon)
                actions.append(a)
                _, r, done, _ = env.step(a)
                if np.random.random() < 0.5:
                    Q1[s, a] += alpha * (r - Q1[s, a])
                else:
                    Q2[s, a] += alpha * (r - Q2[s, a])
            freqs[seed, ep] = np.mean(actions)
    return freqs


freqs_double = run_trap_double(n_episodes=300, alpha=0.1, epsilon=0.1, n_seeds=200)
print(f"Double Q-learning 在 trap 上选 a=1 的频率（最后 50 ep）: {freqs_double[:, -50:].mean():.3f}")
print(f"理论值 ε/2 = 0.05；纯 Q-learning 约 0.25")
print(f"→ Double Q-learning 几乎消除了 maximization bias！")

# 2) 在 CliffWalk 上对比
n_seeds, n_eps = 30, 500
dql_rw = np.zeros((n_seeds, n_eps))
ql_rw = np.zeros((n_seeds, n_eps))
for seed in range(n_seeds):
    env1 = CliffWalk(seed=seed)
    _, r1 = double_q_learning(env1, n_episodes=n_eps, alpha=0.5, epsilon=0.1)
    env2 = CliffWalk(seed=seed)
    _, r2 = q_learning(env2, n_episodes=n_eps, alpha=0.5, epsilon=0.1)
    dql_rw[seed] = r1
    ql_rw[seed] = r2

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(np.convolve(ql_rw.mean(0), np.ones(20)/20, mode='valid'), label='Q-learning', linewidth=2)
ax.plot(np.convolve(dql_rw.mean(0), np.ones(20)/20, mode='valid'), label='Double Q-learning', linewidth=2)
ax.set_xlabel('episode (smoothed w=20)'); ax.set_ylabel('reward')
ax.set_title('CliffWalk: Double Q-learning 方差更小、更稳')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n最终 50 episodes 平均奖励：")
print(f"  Q-learning:        {ql_rw[:, -50:].mean():.2f}")
print(f"  Double Q-learning: {dql_rw[:, -50:].mean():.2f}")